In [2]:
# roche_lobe_overflow_checks_fixed.py
import math
import rebound
import reboundx

def eggleton_RL_over_a(q: float) -> float:
    q13 = q**(1.0/3.0)
    q23 = q13*q13
    return (0.49*q23)/(0.6*q23 + math.log(1.0 + q13))

def is_close(a, b, rtol=1e-10, atol=1e-12):
    return abs(a-b) <= atol + rtol*max(abs(a), abs(b))

# ---- Build a simple 2-body setup ----
sim = rebound.Simulation()
#sim.G = 1
sim.units=['AU','yr','Msun']
sim.integrator = "ias15"
sim.ri_ias15.epsilon=0
sim.dt= 0.0002
# Donor at x=0, accretor at x=a
a_sep = 10.0
m_d = 1.0
m_a = 1.0
sim.add(m=m_d, x=0.0, y=0.0, z=0.0, vx=0.0, vy=0.0, vz=0.0, r=0.00465)     # donor (index 0)
sim.add(m=m_a, x=a_sep, y=0.0, z=0.0, vx=0.0, vy=0.0, vz=0.0, r=0.00465)  # accretor (index 1)
sim.move_to_com()

# Give a tiny relative transverse velocity to make dt_vel finite & deterministic
sim.particles[1].vy = 1e-6
sim.move_to_com()

rx = reboundx.Extras(sim)
rlmt = rx.load_operator("roche_lobe_mass_transfer")
rx.add_operator(rlmt)

# Required operator params (use floats; indices are 0-based)
rlmt.params["rlmt_donor"]     = 0  # donor is particle 0
rlmt.params["rlmt_accretor"]  = 1  # accretor is particle 1
rlmt.params["rlmt_skip_in_CE"] = 1

# RLOF controls
rlmt.params["rlmt_loss_fraction"] = 0.0
rlmt.params["jloss_mode"] =0

# Substepping
rlmt.params["rlmt_substep_max_dm"] = 10#1e-3
rlmt.params["rlmt_substep_max_dr"] = 10#5e-3
rlmt.params["rlmt_min_substeps"]   = 10#3

# Donor particle parameters (set on the actual donor: index 0)
Hp    = 0.1      # pressure scale height (choose not-too-small to avoid extreme exp)
mdot0 = 1e-3     # reference mass-loss rate (>0)
sim.particles[0].params["rlmt_Hp"]    = float(Hp)
sim.particles[0].params["rlmt_mdot0"] = float(mdot0)

# Make the donor slightly overflow its Roche lobe
q = sim.particles[0].m / sim.particles[1].m
RL_over_a = eggleton_RL_over_a(q)
RL_abs = RL_over_a * a_sep
sim.particles[0].r = RL_abs * 1.03  # 3% overflow

# Sanity prints
print("N =", sim.N,
      "donor=", rlmt.params["rlmt_donor"],
      "accretor=", rlmt.params["rlmt_accretor"])
print("RL =", RL_abs, "R_d =", sim.particles[0].r)

p0, p1 = sim.particles[0], sim.particles[1]
M0_i, M1_i = p0.m, p1.m
Mtot_i = M0_i + M1_i
print("masses before:", M0_i, M1_i)

# One operator step (no orbital evolution needed for this check)

for nstep in range(20000):
    sim.step()

    if sim.N == 1:
        p = sim.particles[0]
        print(f"[step {nstep}] N={sim.N} sole mass={p.m}")
        break  # donor was removed; test achieved its end-state
    else:
        p0 = sim.particles[0]
        p1 = sim.particles[1]
        M0_f, M1_f = p0.m, p1.m
        Mtot_f = M0_f + M1_f
        print (sim.particles[1].a)
        print(f"[step {nstep}] masses: {M0_f:.12g}, {M1_f:.12g}; N={sim.N}")

        assert M0_f < 1.0 and M1_f > 1.0
        assert is_close(Mtot_f, 2.0, atol=1e-12)
        assert is_close(Mtot_f, Mtot_i, atol=1e-12), "Total mass must be conserved for f_loss=0."
        assert is_close(p0.params.get("rlof_active", 0), 1), "rlof_active flag must be 1 on donor."
        assert is_close(p1.params.get("rlof_active", 0), 1), "rlof_active flag must be 1 on accretor."
        assert is_close(p0.params.get("inside_CE", 0), 0), "inside_CE must be 0 (not embedded)."
        assert is_close(p1.params.get("inside_CE", 0), 0), "inside_CE must be 0 (not embedded)."
        
    print("OK: RLOF mass transfer happened and flags/cons. mass are correct.")


N = 2 donor= 0 accretor= 1
RL = 3.789205183804563 R_d = 3.9028813393187
masses before: 1.0 1.0
5.0000000000003055
[step 0] masses: 0.999999376668, 1.00000062333; N=2
OK: RLOF mass transfer happened and flags/cons. mass are correct.
5.0000000000002665
[step 1] masses: 0.99999875333, 1.00000124667; N=2
OK: RLOF mass transfer happened and flags/cons. mass are correct.
5.0000000000001785
[step 2] masses: 0.999998129984, 1.00000187002; N=2
OK: RLOF mass transfer happened and flags/cons. mass are correct.
5.00000000000002
[step 3] masses: 0.999997506631, 1.00000249337; N=2
OK: RLOF mass transfer happened and flags/cons. mass are correct.
4.999999999999775
[step 4] masses: 0.999996883272, 1.00000311673; N=2
OK: RLOF mass transfer happened and flags/cons. mass are correct.
4.99999999999942
[step 5] masses: 0.999996259905, 1.00000374009; N=2
OK: RLOF mass transfer happened and flags/cons. mass are correct.
4.999999999998938
[step 6] masses: 0.999995636531, 1.00000436347; N=2
OK: RLOF mass trans

In [7]:

def eggleton_RL_over_a(q: float) -> float:
    q13 = q ** (1/3)
    q23 = q13 * q13
    return (0.49 * q23) / (0.6 * q23 + math.log(1 + q13))

def is_close(a, b, atol=1e-12, rtol=0.0):
    return abs(a - b) <= (atol + rtol * abs(b))

def tuple_close(t0, t1, atol=1e-14):
    return all(abs(x - y) <= atol for x, y in zip(t0, t1))



def step_once(sim, dt=None):
    if dt is None:
        dt = sim.dt
    sim.integrate(sim.t + dt)
def test1_conservative_transfer():
    """
    R_d > R_L so RLOF is active.
    With f_loss=0: donor mass decreases, accretor mass increases by same amount,
    total mass conserved; rlof_active=1, inside_CE=0.
    """
    q = 1.0
    RL_over_a = eggleton_RL_over_a(q)      # ~0.379 for q=1
    R_d = RL_over_a + 0.04                 # mild overflow; still r(=1) > R_d, so no CE

    sim = make_binary_and_rlmt(R_d=R_d, Hp=0.02, mdot0=1e-3,
                               loss_fraction=0.0, jloss_mode=0, dt=1.0)
    p0, p1 = sim.particles[0], sim.particles[1]
    M0_i, M1_i = p0.m, p1.m
    Mtot_i = M0_i + M1_i

    step_once(sim)

    M0_f, M1_f = p0.m, p1.m
    Mtot_f = M0_f + M1_f

    assert M0_f < M0_i, "Donor mass should decrease."
    assert M1_f > M1_i, "Accretor mass should increase."
    assert is_close(Mtot_f, Mtot_i, atol=1e-12), "Total mass must be conserved for f_loss=0."
    assert is_close(p0.params.get("rlof_active", 0.0), 1.0)
    assert is_close(p1.params.get("rlof_active", 0.0), 1.0)
    assert is_close(p0.params.get("inside_CE", 0.0), 0.0)
    assert is_close(p1.params.get("inside_CE", 0.0), 0.0)

def test2_pure_donor_wind_mode0_f1():
    """
    R_d > R_L => RLOF active, but fully non-conservative (f_loss=1)
    with donor-wind (jloss_mode=0). Velocities remain unchanged; accretor mass unchanged.
    """
    q = 1.0
    RL_over_a = eggleton_RL_over_a(q)
    R_d = RL_over_a + 0.04

    sim = make_binary_and_rlmt(R_d=R_d, Hp=0.02, mdot0=1e-3,
                               loss_fraction=1.0, jloss_mode=0, dt=1.0)
    p0, p1 = sim.particles[0], sim.particles[1]
    M0_i, M1_i = p0.m, p1.m
    v0_i = (p0.vx, p0.vy, p0.vz)
    v1_i = (p1.vx, p1.vy, p1.vz)

    step_once(sim)

    M0_f, M1_f = p0.m, p1.m
    v0_f = (p0.vx, p0.vy, p0.vz)
    v1_f = (p1.vx, p1.vy, p1.vz)

    assert M0_f < M0_i, "Donor mass should decrease."
    assert is_close(M1_f, M1_i, atol=1e-14), "Accretor mass should not grow for f_loss=1, mode 0."
    assert tuple_close(v0_f, v0_i, atol=1e-15), "Donor velocity should remain unchanged."
    assert tuple_close(v1_f, v1_i, atol=1e-15), "Accretor velocity should remain unchanged."
    assert is_close(p0.params.get("rlof_active", 0.0), 1.0)
    assert is_close(p1.params.get("rlof_active", 0.0), 1.0)
    assert is_close(p0.params.get("inside_CE", 0.0), 0.0)
    assert is_close(p1.params.get("inside_CE", 0.0), 0.0)

def test3_underfill_suppressed():
    """
    Underfilling case: (R_d - R_L)/Hp << 0 -> exponentially suppressed transfer.
    Expect |ΔM| over one step to be tiny; rlof_active=0.
    """
    q = 1.0
    RL_over_a = eggleton_RL_over_a(q)

    Hp = 1e-3
    R_d = RL_over_a - 25 * Hp     # (R_d - R_L)/Hp = -25

    sim = make_binary_and_rlmt(R_d=R_d, Hp=Hp, mdot0=1e-6,
                               loss_fraction=0.0, jloss_mode=0, dt=1.0)
    p0, p1 = sim.particles[0], sim.particles[1]
    M0_i, M1_i = p0.m, p1.m

    step_once(sim)

    M0_f, M1_f = p0.m, p1.m
    dM_d = abs(M0_f - M0_i)
    dM_a = abs(M1_f - M1_i)

    assert dM_d < 1e-12, f"Underfill should suppress donor mass loss (got {dM_d:g})."
    assert dM_a < 1e-12, f"Underfill should suppress accretion (got {dM_a:g})."
    assert is_close(p0.params.get("rlof_active", 0.0), 0.0)
    assert is_close(p1.params.get("rlof_active", 0.0), 0.0)
    assert is_close(p0.params.get("inside_CE", 0.0), 0.0)
    assert is_close(p1.params.get("inside_CE", 0.0), 0.0)


SyntaxError: '(' was never closed (84376482.py, line 12)